# 01 — Data Overview & Quality Foundation

This notebook establishes the analytical foundation for a NOICE-style streaming platform EDA project. The goal is to verify whether the synthetic raw data is trustworthy enough for downstream content, segmentation, retention, and growth analysis.

## Business Context

Before interpreting listening behavior, product and growth teams need confidence in the source data. This chapter answers foundational questions: what data exists, how complete it is, whether keys can be joined, and which engineered features are available for future analysis.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
sys.path.append(str(PROJECT_ROOT))

from data.process_data_foundation import build_master_dataset

## Step 1 — Inspect Raw Synthetic Data

The raw layer contains separate entity/event tables. We inspect each file for shape, schema, sample records, null values, and duplicate rows before applying transformations.

In [ ]:
raw_tables = {path.stem: pd.read_csv(path) for path in sorted(RAW_DIR.glob("*.csv"))}
print(f"Loaded {len(raw_tables)} raw CSV files from {RAW_DIR}")
for name, df in raw_tables.items():
    print(f"\n{name}: {df.shape[0]:,} rows × {df.shape[1]:,} columns")

In [ ]:
for name, df in raw_tables.items():
    print("=" * 90)
    print(f"TABLE: {name}")
    print("Shape:", df.shape)
    print("\nDtypes:")
    print(df.dtypes)
    print("\nSample rows:")
    display(df.head())
    print("\nMissing values:")
    display(df.isna().sum().to_frame("missing_count").assign(missing_pct=lambda x: x["missing_count"] / len(df)))
    print(f"Duplicate rows: {df.duplicated().sum():,}")

### Initial Data Quality Notes

- Raw files are expected to represent users, content metadata, interaction events, and search-demand context.
- Timestamp columns initially arrive as strings and must be parsed before time-series or retention analysis.
- Interaction events are the grain of the final master dataset; user, content, and demand fields enrich each event.
- Duplicate and missing-value checks protect against inflated playtime, biased completion rates, and broken cohort metrics.

## Step 2 — Clean, Standardize, and Merge

The processing pipeline standardizes column names, parses timestamps, removes duplicates, handles missing values using a 30% threshold rule, and adds categorical code columns for analysis/model-readiness.

In [ ]:
master_df = build_master_dataset()
master_path = PROCESSED_DIR / "master_dataset.csv"
quality_report_path = PROCESSED_DIR / "quality_report.md"

print(f"Master dataset saved to: {master_path}")
print(f"Quality report saved to: {quality_report_path}")
print(f"Master shape: {master_df.shape[0]:,} rows × {master_df.shape[1]:,} columns")

## Shape & Schema Summary

This section verifies that the processed dataset is analysis-ready and contains both raw business fields and engineered behavioral features.

In [ ]:
schema_summary = pd.DataFrame({
    "column": master_df.columns,
    "dtype": master_df.dtypes.astype(str).values,
    "non_null_count": master_df.notna().sum().values,
    "missing_count": master_df.isna().sum().values,
    "missing_pct": master_df.isna().mean().round(4).values,
    "unique_values": master_df.nunique(dropna=True).values,
})
schema_summary

In [ ]:
display(master_df.head())
print(master_df.info())

## Missing Value Heatmap

A missingness heatmap makes it easy to spot systematic gaps. A mostly solid block indicates strong data readiness; vertical stripes would indicate columns needing remediation or exclusion.

In [ ]:
plt.figure(figsize=(14, 6))
sns.heatmap(master_df.isna(), cbar=False, yticklabels=False, cmap="viridis")
plt.title("Missing Value Heatmap — Processed Master Dataset")
plt.xlabel("Columns")
plt.ylabel("Rows")
plt.tight_layout()

## Step 3 — Feature Engineering Check

The master dataset includes session, completion, daypart, weekday, tenure, and power-user features. These fields enable the next notebooks to focus on interpretation rather than repetitive wrangling.

In [ ]:
engineered_columns = [
    "session_duration_minutes",
    "is_completed",
    "time_of_day",
    "day_of_week",
    "user_tenure_days",
    "is_power_user",
]
master_df[engineered_columns].head(10)

In [ ]:
feature_summary = master_df[engineered_columns].agg(["count", "nunique"]).T
feature_summary

## Numeric Distributions

These distributions provide the first read on behavioral scale: how long people listen, how complete sessions are, and whether there are extreme users or content items that may shape business priorities.

In [ ]:
numeric_columns = [
    column for column in [
        "listened_minutes",
        "completion_rate",
        "duration_minutes",
        "session_duration_minutes",
        "user_tenure_days",
        "monthly_searches",
        "content_supply_items",
    ] if column in master_df.columns
]

fig, axes = plt.subplots(nrows=int(np.ceil(len(numeric_columns) / 2)), ncols=2, figsize=(14, 4 * int(np.ceil(len(numeric_columns) / 2))))
axes = np.array(axes).reshape(-1)
for ax, column in zip(axes, numeric_columns):
    sns.histplot(master_df[column], kde=True, ax=ax)
    ax.set_title(f"Distribution of {column}")
for ax in axes[len(numeric_columns):]:
    ax.remove()
plt.tight_layout()

## Top Users and Content Items

Ranking users and content by playtime reveals early concentration patterns. This is useful for identifying power listeners, standout content, and potential creator/content strategy opportunities.

In [ ]:
top_users = (
    master_df.groupby("user_id")
    .agg(
        total_playtime_minutes=("listened_minutes", "sum"),
        sessions=("session_id", "nunique"),
        content_items=("content_id", "nunique"),
        avg_completion_rate=("completion_rate", "mean"),
        is_power_user=("is_power_user", "max"),
    )
    .sort_values("total_playtime_minutes", ascending=False)
    .head(10)
)
top_users

In [ ]:
top_content = (
    master_df.groupby(["content_id", "title", "format", "genre"], dropna=False)
    .agg(
        total_playtime_minutes=("listened_minutes", "sum"),
        listeners=("user_id", "nunique"),
        plays=("event_id", "count"),
        avg_completion_rate=("completion_rate", "mean"),
        skip_rate=("skipped", "mean"),
    )
    .sort_values("total_playtime_minutes", ascending=False)
    .head(10)
)
top_content

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(data=top_users.reset_index(), x="total_playtime_minutes", y="user_id", ax=axes[0], color="#4C78A8")
axes[0].set_title("Top 10 Users by Playtime")
axes[0].set_xlabel("Total Playtime Minutes")
axes[0].set_ylabel("User ID")

sns.barplot(data=top_content.reset_index(), x="total_playtime_minutes", y="content_id", ax=axes[1], color="#F58518")
axes[1].set_title("Top 10 Content Items by Playtime")
axes[1].set_xlabel("Total Playtime Minutes")
axes[1].set_ylabel("Content ID")
plt.tight_layout()

## Data Quality Findings Summary

The generated quality report is the handoff artifact for stakeholders and future notebooks. It documents raw table profiles, cleaning actions, processed dataset shape, and remaining quality risks.

In [ ]:
quality_report = quality_report_path.read_text(encoding="utf-8")
print(quality_report[:4000])

## Phase 1 Handoff

Outputs created by this notebook and pipeline:

- `data/processed/master_dataset.csv` — cleaned, merged, feature-engineered event-level dataset.
- `data/processed/quality_report.md` — markdown report covering schema, missingness, duplicates, and cleaning actions.

The dataset is now ready for content consumption analysis, user segmentation, retention/drop-off analysis, and growth opportunity discovery.